# Reproduce the iMSE benchmark in Colab

Choose a dataset and optionally one backbone, then run the cells in order. The default `STAGE="all"` downloads data, extracts features, and evaluates the full benchmark. Use `STAGE="evaluate"` to reuse completed features.

The repository is loaded from an uploaded ZIP, without connecting to the authors' repository. Model weights and datasets are downloaded from public sources and are not included in this ZIP. OpenOOD checkpoint downloads can be temporarily unavailable because of provider quotas. Verified downloads are retained for retries. ImageNet-1K requires accepted [access terms](https://huggingface.co/datasets/ILSVRC/imagenet-1k) and your own `HF_TOKEN`. Full ImageNet-1K requires substantial disk and compute.

Command output and errors are displayed and saved under `WORK/logs`. Verified data and feature caches are reused. After changing code, use a new results directory while retaining the existing feature cache, as explained in the README.

Select a GPU in **Runtime → Change runtime type**. Add `HF_TOKEN` to **Colab Secrets** for gated downloads. `/content` is temporary; download your results before deleting the runtime or enable optional Drive storage below.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DATASET = "CIFAR10"  # CIFAR10, CIFAR100, ImageNet1K
BACKBONE = ""  # Empty: all paper backbones; e.g. ResNet18 for CIFAR10
STAGE = "all"  # all, download, extract, evaluate
DEVICE = "cuda"  # cpu is also supported except for Vim-Base

ARCHIVE = None  # Leave empty to choose the repository ZIP with the upload button.
PACKAGE_DIR = Path("/content/.imse-source")
WORK = Path("/content/iMSE-work")
SAVE_TO_DRIVE = False  # Optional persistent output storage

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/iMSE-work")


## Upload the repository ZIP

Open this notebook using **File → Upload notebook** in Colab. Run the cell below and choose `iMSE_Anonymous.zip`. Upload the original ZIP, not a folder. No separate dataset or checkpoint uploads are needed. Rerunning the cell reuses the same archive unless you reset `ARCHIVE=None`.


In [ ]:
import hashlib
import shutil
import stat
import zipfile
from pathlib import PurePosixPath

def unpack_repository(archive, destination):
    archive = Path(archive).expanduser().resolve()
    if not archive.is_file():
        raise FileNotFoundError(f"Upload the repository ZIP first, then set ARCHIVE to its path: {archive}")
    with archive.open("rb") as handle:
        digest = hashlib.file_digest(handle, "sha256").hexdigest()
    target = (Path(destination) / digest[:16]).resolve()
    with zipfile.ZipFile(archive) as bundle:
        members = bundle.infolist()
        names = set()
        for entry in members:
            name = entry.filename
            parts = PurePosixPath(name).parts
            if (not parts or name.startswith("/") or "\\" in name or ":" in name
                    or ".." in parts or ".git" in parts
                    or stat.S_ISLNK(entry.external_attr >> 16) or name in names):
                raise ValueError(f"Unsafe or duplicate ZIP member: {name}")
            names.add(name)
        roots = [PurePosixPath(name).parent for name in names
                 if name.endswith("pyproject.toml")
                 and (PurePosixPath(name).parent / "imse_benchmark/__init__.py").as_posix() in names]
        if len(roots) != 1:
            raise ValueError("Expected one iMSE repository in the ZIP.")
        target.mkdir(parents=True, exist_ok=True)
        for entry in members:
            if entry.is_dir():
                continue
            output = target.joinpath(*PurePosixPath(entry.filename).parts)
            output.parent.mkdir(parents=True, exist_ok=True)
            if output.is_file():
                with output.open("rb") as saved, bundle.open(entry) as source:
                    if hashlib.file_digest(saved, "sha256").digest() != hashlib.file_digest(source, "sha256").digest():
                        raise RuntimeError(f"Extracted file differs from the uploaded ZIP: {output}. Use a new PACKAGE_DIR.")
            else:
                temporary = output.with_name(output.name + ".partial")
                with bundle.open(entry) as source, temporary.open("wb") as saved:
                    shutil.copyfileobj(source, saved)
                temporary.replace(output)
    print("Repository ZIP SHA-256:", digest)
    return target.joinpath(*roots[0].parts)

if ARCHIVE is None:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
        raise ValueError("Upload exactly one repository ZIP.")
    ARCHIVE = Path(next(iter(uploaded))).resolve()
    del uploaded
REPO = unpack_repository(ARCHIVE, PACKAGE_DIR)
print("Repository:", REPO)


In [ ]:
def run(command, label):
    from datetime import datetime, timezone

    log_path = WORK / "logs" / f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}-{label}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env.setdefault("TQDM_MININTERVAL", "10")
    print(f"{label} | log: {log_path}", flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=REPO if REPO.exists() else None,
                                   env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace")
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
        finally:
            process.stdout.close()
    if code:
        raise RuntimeError(f"{label} exited with code {code}. See the output above or {log_path}.")


## Install and check the environment

In [ ]:
from google.colab import userdata

if not os.environ.get("HF_TOKEN"):
    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
    if token:
        os.environ["HF_TOKEN"] = token
    del token

needs_vim = (DATASET == "ImageNet1K" and BACKBONE in {"", "Vim-Base"}
             and STAGE in {"all", "extract"})
package = f"{REPO}[vim]" if needs_vim else str(REPO)
run([sys.executable, "-m", "pip", "install", "-e", package], "install")

# Inspect the manifest without importing packages that pip may have just updated.
import json
checkpoints = json.loads((REPO / "imse_benchmark" / "assets" / "checkpoints.json").read_text())
models = [key.split("/", 1)[1] for key in checkpoints if key.startswith(DATASET + "/")]
if STAGE not in {"all", "download", "extract", "evaluate"} or not models:
    raise ValueError("Choose a valid DATASET and STAGE.")
if BACKBONE and BACKBONE not in models:
    raise ValueError(f"Choose a backbone from {models}, or leave BACKBONE empty.")
print("Backbones:", models if not BACKBONE else [BACKBONE])
print("Work directory:", WORK)
run([sys.executable, "-m", "pip", "show", "torch", "torchvision", "timm"], "versions")
if STAGE != "download":
    check = "import torch, torchvision, timm, faiss; print('PyTorch:', torch.__version__); "
    if DEVICE.startswith("cuda"):
        check += "assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(0))"
    run([sys.executable, "-c", check], "environment")


## Download datasets

Downloads the original benchmark archives automatically. If an OpenOOD public Drive link is unavailable, a pinned public mirror is used and its SHA-256 is verified. No manual upload or access to the authors' Drive is required. Completed verified downloads are reused. ImageNet-1K requires accepted dataset access terms and your own `HF_TOKEN`.


In [ ]:
if STAGE in {"all", "download"}:
    args = [sys.executable, "-u", "-m", "imse_benchmark", "download", "--dataset", DATASET,
            "--work-dir", str(WORK), "--device", DEVICE]
    if BACKBONE:
        args += ["--backbone", BACKBONE]
    run(args, DATASET + "-download")

## Extract features

Loads the released checkpoints and retains the original preprocessing, split order, and extraction batch sizes. Completed splits are reused.

In [ ]:
if STAGE in {"all", "extract"}:
    args = [sys.executable, "-u", "-m", "imse_benchmark", "extract", "--dataset", DATASET,
            "--work-dir", str(WORK), "--device", DEVICE]
    if BACKBONE:
        args += ["--backbone", BACKBONE]
    run(args, DATASET + "-extract")

## Evaluate detectors

Fits on ID training data and evaluates every paper detector. Prints All, Near, and Far tables, with metrics saved under `WORK/results`.

In [ ]:
if STAGE in {"all", "evaluate"}:
    args = [sys.executable, "-u", "-m", "imse_benchmark", "evaluate", "--dataset", DATASET,
            "--work-dir", str(WORK), "--device", DEVICE]
    if BACKBONE:
        args += ["--backbone", BACKBONE]
    run(args, DATASET + "-evaluate")